# MaxCut — QAOA (PennyLane, estilo demo)

Implementación siguiendo el tutorial
[QAOA for MaxCut](https://pennylane.ai/demos/tutorial_qaoa_maxcut)
(Farhi–Goldstone–Gutmann): capas $U_C(\gamma)\,U_B(\beta)$ sobre el grafo,
optimización clásica de parámetros y muestreo del bitstring.

Grafo: **HamLib** `hamlib-maxcut` índice 8 ($n=6$, `circ_offsets-1-2`), no el cuadrado de 4 nodos del demo —
mismo circuito, otra instancia de benchmark.


In [1]:
import itertools
import json
import math
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

import pennylane as qml
from pennylane import numpy as pnp


In [2]:
def resolve_maxcut_dir() -> Path:
    for base in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        root = base / "Notebooks" / "Optimization" / "MaxCutOptimization"
        if root.is_dir():
            return root
    return Path.cwd().resolve() / "Notebooks" / "Optimization" / "MaxCutOptimization"


MAXCUT_DIR = resolve_maxcut_dir()
INSTANCE_PATH = MAXCUT_DIR / "instance" / "hamlib_maxcut.json"
METRICS_DIR = MAXCUT_DIR / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

instance = json.loads(INSTANCE_PATH.read_text())
n = int(instance["n"])
edges = [tuple(e) for e in instance["edges"]]
OPTIMAL_CUT = int(instance["optimal_cut"])
OPTIMAL_BITS = list(instance["optimal_bits"])
nodes = list(range(n))

print(
    f"HamLib MaxCut | index={instance['hamlib_index']} id={instance['hamlib_id']} "
    f"n={n} |E|={len(edges)} optimal_cut={OPTIMAL_CUT}"
)
print("Fuente:", instance["url"])
print("Demo QAOA:", instance["demo_ref"])


def cut_value(bits) -> int:
    return int(sum(1 for a, b in edges if bits[a] != bits[b]))


def is_success(bits) -> bool:
    return cut_value(bits) == OPTIMAL_CUT


def save_maxcut_metrics(payload: dict, filename: str):
    path = METRICS_DIR / filename
    path.write_text(json.dumps(payload, indent=2))
    print("Saved:", path)
    return path


HamLib MaxCut | index=8 id=circ_offsets-1-2 n=6 |E|=12 optimal_cut=8
Fuente: https://pennylane.ai/datasets/hamlib-maxcut
Demo QAOA: https://pennylane.ai/demos/tutorial_qaoa_maxcut


## Circuito QAOA (demo PennyLane)

$U_B(\beta)=\prod_j R_X(2\beta)_j$,\;
$U_C(\gamma)=\prod_{(j,k)\in E}\mathrm{CNOT}\,R_Z(\gamma)\,\mathrm{CNOT}$.

Objetivo: minimizar $-C$ con $C=\tfrac12\big(|E|-\langle\sum Z_j Z_k\rangle\big)$.


In [3]:
N_LAYERS = 2
N_RUNS = 8
STEPS = 40
SHOTS_SAMPLE = 200

def U_B(beta):
    for wire in range(n):
        qml.RX(2 * beta, wires=wire)

def U_C(gamma):
    for edge in edges:
        qml.CNOT(wires=edge)
        qml.RZ(gamma, wires=edge[1])
        qml.CNOT(wires=edge)

dev = qml.device("default.qubit", wires=n)

@qml.qnode(dev)
def circuit(gammas, betas, return_samples=False):
    for wire in range(n):
        qml.Hadamard(wires=wire)
    for gamma, beta in zip(gammas, betas):
        U_C(gamma)
        U_B(beta)
    if return_samples:
        return qml.sample()
    zz = qml.sum(*(qml.Z(a) @ qml.Z(b) for a, b in edges))
    return qml.expval(zz)

def objective(params):
    gammas, betas = params[0], params[1]
    return -0.5 * (len(edges) - circuit(gammas, betas))

def most_frequent_bits(params, shots=SHOTS_SAMPLE):
    samples = qml.set_shots(circuit, shots=shots)(params[0], params[1], return_samples=True)
    # samples shape (shots, n)
    as_tuples = [tuple(int(b) for b in row) for row in np.array(samples)]
    counts = {}
    for t in as_tuples:
        counts[t] = counts.get(t, 0) + 1
    best = max(counts, key=counts.get)
    return list(best), counts

def run_qaoa(seed: int):
    pnp.random.seed(seed)
    random.seed(seed)
    init = 0.01 * pnp.random.rand(2, N_LAYERS, requires_grad=True)
    opt = qml.AdagradOptimizer(stepsize=0.5)
    params = init
    t0 = time.perf_counter()
    for _ in range(STEPS):
        params = opt.step(objective, params)
    bits, _ = most_frequent_bits(params)
    elapsed = time.perf_counter() - t0
    c = cut_value(bits)
    return {
        "seed": seed,
        "bits": bits,
        "cut": float(c),
        "success": bool(is_success(bits)),
        "time_s": float(elapsed),
        "energy_evals": STEPS,  # opt steps (expval calls)
        "params_gamma": [float(x) for x in np.array(params[0]).ravel()],
        "params_beta": [float(x) for x in np.array(params[1]).ravel()],
    }

rows = []
for seed in range(N_RUNS):
    out = run_qaoa(seed)
    rows.append(out)
    print(f"seed={seed:02d} success={out['success']} cut={out['cut']} time={out['time_s']:.2f}s bits={out['bits']}")

cuts = np.array([r["cut"] for r in rows], dtype=float)
times = np.array([r["time_s"] for r in rows], dtype=float)
success_rate = float(np.mean([r["success"] for r in rows]))
print(f"success_rate={success_rate:.3f}")

payload = {
    "problem": "maxcut",
    "solver": "qaoa",
    "family": "quantum_inspired",
    "protocol": {
        "n_runs": N_RUNS,
        "p": N_LAYERS,
        "opt_steps": STEPS,
        "optimizer": "Adagrad",
        "backend": "pennylane_default.qubit",
        "demo_ref": instance["demo_ref"],
        "sense": "maximize_cut",
        "hamlib_index": instance["hamlib_index"],
        "hamlib_id": instance["hamlib_id"],
        "shots_sample": SHOTS_SAMPLE,
    },
    "instance": {"n": n, "edges": [list(e) for e in edges], "source": instance["source"]},
    "quality": {
        "optimal_cost": float(OPTIMAL_CUT),
        "optimal_bits": OPTIMAL_BITS,
        "cost_mean": float(np.mean(cuts)),
        "cost_best": float(np.max(cuts)),
        "gap_mean": float(np.mean(OPTIMAL_CUT - cuts)),
    },
    "success_rate": success_rate,
    "time_s": {"mean": float(np.mean(times)), "std": float(np.std(times)), "total": float(np.sum(times))},
    "effort": {"energy_evals_mean": float(np.mean([r["energy_evals"] for r in rows]))},
    "runs": rows,
}
save_maxcut_metrics(payload, "qaoa_maxcut.json")


seed=00 success=True cut=8.0 time=0.65s bits=[1, 0, 1, 1, 0, 1]
seed=01 success=True cut=8.0 time=0.60s bits=[0, 0, 1, 0, 0, 1]
seed=02 success=True cut=8.0 time=0.61s bits=[0, 0, 1, 0, 0, 1]
seed=03 success=True cut=8.0 time=0.83s bits=[1, 0, 1, 1, 0, 1]
seed=04 success=True cut=8.0 time=0.62s bits=[1, 1, 0, 1, 1, 0]
seed=05 success=True cut=8.0 time=0.62s bits=[1, 0, 0, 1, 0, 0]
seed=06 success=True cut=8.0 time=0.61s bits=[0, 1, 0, 0, 1, 0]
seed=07 success=True cut=8.0 time=0.81s bits=[1, 1, 0, 1, 1, 0]
success_rate=1.000
Saved: /Users/juanprada/Documents/UNIR-MasterQC-QuantumAdvantageInDataScience/Notebooks/Optimization/MaxCutOptimization/metrics/qaoa_maxcut.json


PosixPath('/Users/juanprada/Documents/UNIR-MasterQC-QuantumAdvantageInDataScience/Notebooks/Optimization/MaxCutOptimization/metrics/qaoa_maxcut.json')